# M01 · Python for earthquake catalogs

**Week 1 · about 9.8 h (+ 1 h optional)** · before this: M00

> **한국어 요약** 지진 목록을 다루는 데 필요한 파이썬 기초(변수, 리스트, 딕셔너리, 반복문, 함수, 오류 메시지 읽기)를 익힙니다. NumPy로 여러 지점 사이의 대원 거리를 한 번에 계산하고, pandas로 지진 목록 파일을 읽어 UTC 시각으로 맞춘 뒤 거리와 시간으로 사건을 고릅니다. 마지막으로 UTC 시각을 KST로 바꾸는 두 방법을 비교해, 오래된 기록에서 한 시간이 어긋나는 사건을 찾습니다.

**You will**

- Write a vectorized NumPy function for great-circle distances
- Convert KST clock readings to UTC across date changes
- Load, select and sort catalogs with pandas
- Compare two ways of turning UTC times into KST

**New words:** epicenter (진앙), hypocenter (진원), traceback (트레이스백), vectorization (벡터화), great-circle distance (대원 거리), aware datetime (인식 시각, 시간대 정보가 있는 시각), DataFrame (데이터프레임), time zone (시간대) · [all terms](../../../docs/glossary.md)

**Optional reading:** Programming with Python, lists, loops, functions, errors; Plotting and Programming in Python, reading tables and DataFrames

Log any AI help in `work/ai_log/M01.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M01.yaml](../../../.sgtl/criteria/M01.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
EXAMPLE_LAT = 35.76        # degrees north: the 2016 Gyeongju epicenter area (Part 4)
EXAMPLE_LON = 129.19       # degrees east
EXAMPLE_RADIUS_KM = 15.0   # search radius in kilometers (Part 4)
MIN_ML = 3.0               # smallest local magnitude in the summary tables
N_SHOW = 5                 # how many rows head() and tail() show
DEMO_SEED = 2016           # random seed of the speed demo (Part 2)
N_DEMO = 1_000_000         # number of values in the speed demo (Part 2)

In [ ]:
import sgtl_tutorial as st
S = st.init("M01")

import datetime as dt
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Image, display


def show_figure(figure, filename):
    """Save and display the same static figure, including in an archived notebook."""
    path = Path("figures") / filename
    path.parent.mkdir(exist_ok=True)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    display(Image(filename=str(path)))
    plt.close(figure)

## Part 1 · Python in context (about 90 min)

A **variable** is a name for a value. Every value has a **type**: `str` (text), `int` (whole number), `float`
(decimal number) or `bool` (`True` or `False`). The cell keeps one row of the KMA catalog in variables: the 2016
Gyeongju **mainshock**, the largest earthquake of its sequence.

In [ ]:
event_id = "2016000291"   # str: an ID is a label, so keep it as text
latitude = 35.76          # float, degrees north
longitude = 129.19        # float, degrees east
depth_km = 15.0           # float, kilometers
ml = 5.8                  # float, local magnitude
year = 2016               # int
print(type(event_id), type(ml), type(year))
print(int("0042"), "versus", "0042")   # a number forgets leading zeros; text keeps them

The **hypocenter** is the point inside the Earth where the rupture starts; the **epicenter** is the point on the
surface right above it. The **focal depth** is the depth of the hypocenter, in km. The local magnitude ML comes from
the largest amplitudes on nearby seismometers. `st.glossary.term` prints any term of the glossary:

In [ ]:
print(st.glossary.term("hypocenter"))
print(st.glossary.term("local_magnitude"))

The same earthquake in two views: the map shows the epicenter, the section the hypocenter below it. The sketch uses
the variables above; it is not a measured fault. M02 shows how to build such figures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
axes[0].plot(longitude, latitude, "*", color="crimson", ms=17)
axes[0].annotate("Epicenter", (longitude, latitude), xytext=(12, 14), textcoords="offset points")
axes[0].set(xlim=(longitude - 0.15, longitude + 0.15), ylim=(latitude - 0.15, latitude + 0.15),
            xlabel="Longitude (°E)", ylabel="Latitude (°N)", title="Surface position: map view")
axes[1].axhline(0, color="0.2", lw=2)
axes[1].plot([0, 0], [0, depth_km], "--", color="0.5")
axes[1].plot(0, 0, "*", color="crimson", ms=14)
axes[1].plot(0, depth_km, "*", color="tab:blue", ms=14)
axes[1].annotate("Hypocenter", (0, depth_km), xytext=(12, 0), textcoords="offset points")
axes[1].set(xlim=(-10, 10), ylim=(22, -2), xlabel="Horizontal offset from epicenter (km)",
            ylabel="Depth below surface (km)", title="Source position: schematic section")
for ax in axes:
    ax.grid(alpha=0.2)
show_figure(fig, "M01_epicenter_hypocenter.png")

A **list** keeps values in order. Positions start at 0, negative positions count from the end, and `a[i:j]` takes
positions `i` up to, but not including, `j`. A **dict** maps keys to values, like one row of a table.

In [ ]:
mags = [2.3, 5.1, 3.0, 5.8, 2.7]   # practice numbers
print(mags[0], mags[-1], len(mags), mags[1:3])
mags.append(4.5)
print(mags)

event = {"event_id": "2016000291", "ml": 5.8, "depth_km": 15.0}
event["region"] = "Gyeongju"
print(event["ml"], list(event.keys()))

A **loop** (`for ... in ...:`) runs the indented lines once for every item; `if` runs them only when a test is
`True`.

In [ ]:
n_big = 0
for m in mags:
    if m >= 5.0:
        n_big += 1
print(f"{n_big} of {len(mags)} magnitudes are 5.0 or larger")

A **function** takes inputs (arguments) and returns an output. Its **docstring**, the text in triple quotes under
`def`, says what goes in and what comes out, with units; `help()` prints it. An **f-string** (`f"..."`) puts values
into text. `{x:.2f}` gives two decimals, `{n:03d}` pads a whole number with zeros to 3 digits, and `{s:>6}` aligns
text right in 6 characters.

In [ ]:
def ml_label(magnitude, decimals=1):
    """Short text label for a local magnitude.

    magnitude : float, local magnitude ML
    decimals  : int, number of decimal places (default 1)
    returns   : str, for example 'ML 5.8'
    """
    return f"ML {magnitude:.{decimals}f}"


print(ml_label(5.8), "|", ml_label(2.25, decimals=2))
print(f"[{7:03d}] [{3.14159:.2f}] [{'KS':>6}]")
help(ml_label)

When Python stops with an error, it prints a **traceback**: the chain of calls that led to the error. Read it
**from the bottom up**. The last line names the error; the lines above show where Python was, call by call. The
cell makes an error on purpose and prints the traceback without stopping the notebook.

In [ ]:
def mean_magnitude(values):
    """Average of a list of magnitudes."""
    return sum(values) / len(values)


def describe(values):
    """One line of text about a list of magnitudes."""
    return f"mean {mean_magnitude(values):.2f} from {len(values)} events"


print(describe(mags))
try:
    print(describe([]))   # an empty list: what goes wrong, and where?
except ZeroDivisionError:
    traceback.print_exc()

The last line says `ZeroDivisionError: division by zero`. Above it, the traceback points at
`return sum(values) / len(values)` in `mean_magnitude`, which `describe` called: the list was empty. **Try it:** run
`describe(["5.8"])` in a new cell (**+** in the toolbar adds one) and read its traceback the same way.

## Part 2 · NumPy arrays and vectorization (about 45 min, then task 1)

A NumPy **array** holds numbers of one type and does math on all of them at once. That is **vectorization**: no
loop over the items. A single number is used for every item (**broadcasting**). A comparison gives a **boolean
array**, one `True` or `False` per item, and `sum()` counts the `True` values.

In [ ]:
depths = np.array([15.0, 12.0, 8.5, 14.0])
print(depths.dtype, depths.shape)
print(depths * 1000.0)             # kilometers to meters, for every item at once
print(depths > 10.0)               # a boolean array
print((depths > 10.0).sum(), "depths are deeper than 10 km")

NumPy's `sin`, `cos` and `arcsin` work in **radians**: a full circle is 2π radians, or 360°. `np.radians` turns
degrees into radians and `np.degrees` turns them back. `np.clip` keeps values inside a range.

In [ ]:
angles_deg = np.array([0.0, 30.0, 90.0])
print(np.sin(angles_deg))               # wrong on purpose: 30 is read as 30 radians
print(np.sin(np.radians(angles_deg)))   # right: 0, 0.5, 1
print(np.degrees(np.arcsin(0.5)))       # and back to degrees: 30
print(np.sqrt(np.array([0.0, 0.25, 1.0])))
print(np.clip(np.array([-0.1, 0.5, 1.0000000000000002]), 0.0, 1.0))   # rounding can leave a value just above 1

Why vectorize? Shorter code, and speed. The cell times a Python loop and one array operation on `N_DEMO` random
numbers. A **random seed** fixes the random-number generator: the same seed gives the same numbers. The checks of
this module draw fresh practice cases from your own seed; `st.reroll("M01-T1")` draws new ones.

In [ ]:
rng = np.random.default_rng(DEMO_SEED)   # a random generator fixed by a random seed
x = rng.uniform(0.0, 10.0, N_DEMO)
t0 = time.perf_counter()
y_loop = [value * 2.0 for value in x]
t1 = time.perf_counter()
y_vec = x * 2.0
t2 = time.perf_counter()
print(f"loop {t1 - t0:.3f} s, vectorized {t2 - t1:.4f} s, same result: {np.allclose(y_loop, y_vec)}")

The **great-circle distance** is the shortest distance between two points along the Earth's surface. On a sphere it
is the radius times the **central angle**, the angle between the two points seen from the Earth's center. Do not
treat degrees as flat x and y: a degree of longitude shrinks from the equator toward the poles.

The haversine formula gives the central angle $c$ from latitudes $\varphi$ and longitudes $\lambda$ **in radians**:

$$a = \sin^2\!\left(\frac{\varphi_2 - \varphi_1}{2}\right) + \cos\varphi_1\,\cos\varphi_2\,
  \sin^2\!\left(\frac{\lambda_2 - \lambda_1}{2}\right), \qquad c = 2\arcsin\sqrt{a}, \qquad d = R\,c$$

$d$ is the distance, and $R$ = 6371.0 km is the radius of the sphere that stands for the Earth.

### M01-T1 · Great-circle distance · core · about 90 min

Write `haversine_km(lat1, lon1, lat2, lon2)`: the distance in km between pairs of points, with the formula above. It
takes numpy arrays and gives one distance per pair; one point may be two plain numbers.

Example: `haversine_km(0, 0, 0, 1)` is about 111.195 km, one degree along the equator.

Use numpy functions, no Python loop. Not allowed: a ready-made distance function (pyproj, geopy, geographiclib,
cartopy, pyrocko, scikit-learn, `obspy.geodetics`, `obspy.taup`, the haversine package).

In [ ]:
# Before you code: about how many kilometers apart are the 2016 Gyeongju and 2017 Pohang mainshocks?
st.predict("M01-T1", """

""")

In [ ]:
EARTH_RADIUS_KM = 6371.0   # sphere radius of the haversine formula


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance on a sphere of radius EARTH_RADIUS_KM.

    lat1, lon1 : float or numpy array, first point(s) in decimal degrees (north and east positive)
    lat2, lon2 : float or numpy array, second point(s) in decimal degrees
                 Arrays share one shape; one point given as two floats is paired with every point of the arrays.
    returns    : distance(s) in kilometers, one per pair (numpy array of that shape, or one number for one pair
                 of floats)
    """
    # Step 1: convert the angles you need from degrees to radians (numpy works in radians).
    # Step 2: compute the term a of the haversine formula, for all pairs at once.
    # Step 3: turn a into the central angle, then multiply by the radius.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T1: write your code here")

In [ ]:
st.check("M01-T1", func=haversine_km)

**Explain:** In a new cell, run `haversine_km(35.76, 129.19, 36.11, 129.37)`: the distance from the 2016 Gyeongju
mainshock to the 2017 Pohang mainshock. Does the number fit a map of Korea?

## Part 3 · Dates and time zones (about 45 min, then task 2)

The **origin time** of an earthquake is when the rupture starts. Catalogs and **seismograms**, records of ground
motion at one station, give it in UTC. The KMA announces earthquakes in KST, and **KST = UTC + 9 h, always**. The
`Z` in `2016-09-12T11:32:54Z` means UTC, so the Gyeongju mainshock struck at 20:32:54 KST.

A Python `datetime` is **naive**, a clock reading without a zone, or **aware**, with a zone: one real moment. A
**time zone** turns a moment into the clock reading of a place; a fixed offset such as UTC + 9 h is the simplest.
Adding a `timedelta` does the calendar arithmetic. `strftime` writes a time as text: `%Y` year, `%m` month, `%d`
day, `%H` hour, `%M` minute, `%S` second.

In [ ]:
t = dt.datetime(2019, 7, 6, 3, 19, 53, tzinfo=dt.timezone.utc)   # an aware moment in UTC (2019 Ridgecrest)
print(t.isoformat())
print(t + dt.timedelta(hours=36))          # 36 hours later: the date changes by itself
print(t - dt.timedelta(days=7))            # a week earlier
print(t.strftime("%d/%m/%Y at %H.%M"))     # the same moment as text, in a format of our choice

In [ ]:
ist = dt.timezone(dt.timedelta(hours=5, minutes=30))   # a fixed offset: India Standard Time, UTC + 5:30
print(t.astimezone(ist))                                # the same moment on an Indian clock
naive = dt.datetime(2019, 7, 6, 3, 19, 53)
print(naive.tzinfo, "| aware:", t.tzinfo)               # None means naive: Python does not know the zone
print(dt.datetime(2024, 2, 28, 23, 0) + dt.timedelta(hours=2))   # leap years are handled too

### M01-T2 · KST clock reading to a UTC event ID · core · about 60 min

Write `kst_to_utc_id(year, month, day, hour, minute, second)`: the UTC time ID `'YYYYMMDDHHMMSS'` of a KST clock
reading, the key that our data files use. KST = UTC + 9 h, always: a fixed offset. Going back 9 hours can change the
day, the month and the year.

Example: the Gyeongju mainshock struck at 2016-09-12 20:32:54 KST, so `kst_to_utc_id(2016, 9, 12, 20, 32, 54)`
returns `'20160912113254'`.

In [ ]:
# Before you code: for which KST clock readings is the UTC date the day before?
st.predict("M01-T2", """

""")

In [ ]:
def kst_to_utc_id(year, month, day, hour, minute, second):
    """UTC time ID of a KST clock reading.

    year, month, day, hour, minute, second : int, a date and time of day in KST (UTC + 9 h)
    returns : str, the same moment in UTC written as 'YYYYMMDDHHMMSS' (14 digits, zero-padded, no separators)
    """
    # Step 1: make one datetime from the six numbers.
    # Step 2: go from KST to UTC with a fixed offset.
    # Step 3: write the result as 14 digits.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T2: write your code here")

In [ ]:
st.check("M01-T2", func=kst_to_utc_id)

**Explain:** Why do data files store times in UTC, and convert them to KST only to show them to people?

## Part 4 · pandas: tables of earthquakes (about 60 min, then tasks 3 and 4)

A **DataFrame** is a pandas table with named columns. One column is a **Series**; the row labels are the **index**.
`df["ml"]` picks a column, and `df[df["ml"] >= 3]` picks rows with a boolean array (**boolean filtering**).
`sort_values` orders the rows, and `groupby` summarizes groups of rows. First, the course's station list:

In [ ]:
stations = st.data.stations()
print(stations.shape, "(rows, columns)")
print(stations.dtypes)
stations.head(N_SHOW)

In [ ]:
ks = stations[stations["network"] == "KS"]   # boolean filtering
print(len(ks), "KS stations")
north_kg = stations[(stations["network"] == "KG") & (stations["latitude"] > 37.0)]   # & means and
north_kg.sort_values("latitude", ascending=False).head(N_SHOW)   # sort_values orders the rows

In [ ]:
print(stations.groupby("network").size())   # how many stations each network has
print(stations.groupby("network")["latitude"].agg(["min", "max"]).round(2))

**Blank fields** become **NaN** ("not a number", pandas' mark for a missing value), and the column stays float.
`isna()` finds them. Some programs, such as Excel, start a UTF-8 file with a **byte order mark** (BOM): three
invisible bytes. pandas skips it. Python's `open()` with `"utf-8"` adds it to the first column name as
`'\ufeff'`; `"utf-8-sig"` removes it.

In [ ]:
import csv
import io
import tempfile

small = pd.read_csv(io.StringIO("station,elevation_m\nSTA1,120\nSTA2,\n"))
print(small)
print(small["elevation_m"].isna().tolist(), small["elevation_m"].dtype)

with tempfile.TemporaryDirectory() as folder:
    demo = Path(folder) / "with_bom.csv"
    demo.write_text("station,elevation_m\nSTA1,120\n", encoding="utf-8-sig")   # writes a BOM first
    print(demo.read_bytes()[:3])                    # b'\xef\xbb\xbf' is the BOM
    with open(demo, encoding="utf-8") as fh:
        print(next(csv.reader(fh)))                 # the first name is '\ufeffstation'
    with open(demo, encoding="utf-8-sig") as fh:
        print(next(csv.reader(fh)))                 # clean: 'station'
    print(list(pd.read_csv(demo).columns))          # pandas handles it by itself

**Times in pandas.** `pd.to_datetime(..., utc=True)` turns text into aware UTC times. It also builds times from
columns named year, month, day, hour, minute and second. On a whole column, `.dt.tz_convert(zone)` shows the same
moments on another clock. `.dt.tz_localize(zone)` says which zone naive readings belong to, without a shift, and
`+ pd.Timedelta(hours=...)` shifts them.

In [ ]:
times = pd.to_datetime(pd.Series(["2016-09-12T10:44:32Z", "2016-09-12T11:32:54Z"]), utc=True)
print(times.dt.tz, "|", times.dt.strftime("%H:%M").tolist())
print(times.dt.tz_convert(ist).tolist())                          # the same moments on an Indian clock
clock = pd.Series(pd.to_datetime(["2020-03-01 00:30", "2020-12-31 23:30"]))
print(clock.dt.tz, "->", clock.dt.tz_localize("UTC").tolist())   # naive -> aware, no shift
print((clock + pd.Timedelta(hours=2)).tolist())                  # shift a whole column at once
parts = pd.DataFrame({"year": [2020], "month": [2], "day": [29], "hour": [6], "minute": [0], "second": [0]})
print(pd.to_datetime(parts).tolist())                            # times from columns of date parts

**The KMA catalog.** `st.data.kma_catalog()` loads the KMA catalog of the course. It is used under **KOGL Type 1**,
the Korea Open Government License: anyone may reuse it if they name the source. `cat.attrs["source"]` holds
that text. `origin_utc` is UTC, `origin_kst` shows the same moments at +09:00, and `depth_km` is NaN where KMA gives
no depth.

In [ ]:
cat = st.data.kma_catalog()
print(len(cat), "events from", cat["origin_utc"].min(), "to", cat["origin_utc"].max())
print(cat.attrs["source"].splitlines()[0])
cat.tail(N_SHOW)

In [ ]:
print(cat["depth_km"].isna().sum(), "events have no depth")
big = cat[cat["ml"] >= MIN_ML]
per_year = big.groupby(big["origin_utc"].dt.year).size()
print(f"events with ML >= {MIN_ML}, per year (last 10 years):")
print(per_year.tail(10))

### M01-T3 · Load a colleague's catalog file · core · about 120 min

Write `load_catalog(path, tz)`: read a colleague's CSV file into a clean DataFrame; the docstring lists the columns.
The files have the header `Year,Month,Day,Hour,Minute,Second,Latitude,Longitude,Depth,Magnitude`. But some list
`Magnitude` before `Depth`, some start with a BOM, and `Depth` is blank when unknown. `tz` (`"KST"` or `"UTC"`)
names the clock of the time columns.

Example: with `tz="KST"`, the line `2016,9,12,20,32,54,35.76,129.19,15.0,5.8` gives `origin_utc`
2016-09-12 11:32:54+00:00, `depth_km` 15.0 and `ml` 5.8.

In [ ]:
# Before you code: which quirks of these files will pandas handle by itself, and which will need your own code?
st.predict("M01-T3", """

""")

In [ ]:
def load_catalog(path, tz):
    """Read a catalog CSV with separate time columns into a clean table.

    path    : str or pathlib.Path, a CSV file with the columns Year, Month, Day, Hour, Minute, Second,
              Latitude, Longitude, Depth, Magnitude (Depth and Magnitude in either order, so pick columns by
              name; the file may start with a byte order mark; Depth may be blank)
    tz      : "KST" or "UTC", the clock of the six time columns (KST = UTC + 9 h, a fixed offset)
    returns : pandas DataFrame with exactly the columns origin_utc (aware, in UTC), latitude, longitude,
              depth_km (NaN when blank), ml, in this order; the last four are floats (degrees, km, ML).
              One row per data line, in the file's order.
    """
    # Step 1: read the file; later, pick columns by their names (never by position).
    # Step 2: build times from the six time columns; KST moves back 9 hours; declare the times UTC.
    # Step 3: put the five output columns together, in the docstring's order.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T3: write your code here")

In [ ]:
st.check("M01-T3", func=load_catalog)

**Explain:** The cell above the task counts the catalog's events without a depth. What would 0 instead of NaN for
them do to the mean depth of the catalog?

### M01-T4 · Events within a radius, earliest first · core · about 60 min

Write `events_within(df, lat0, lon0, r_km)`: the rows of `df` whose epicenter lies at most `r_km` from (`lat0`,
`lon0`), earliest `origin_utc` first. Use your `haversine_km`. Keep all columns, and leave `df` unchanged.

Example: `df` holds the Gyeongju mainshock (35.76, 129.19; 11:32:54 UTC), its foreshock (35.77, 129.19; 10:44:32
UTC) and the 2017 Pohang mainshock. `events_within(df, 35.76, 129.19, 15)` returns the foreshock, then the
mainshock.

Not allowed: the distance libraries of task 1.

In [ ]:
# Before you code: about how many catalog events lie within EXAMPLE_RADIUS_KM (15 km) of the Gyeongju mainshock?
st.predict("M01-T4", """

""")

In [ ]:
def events_within(df, lat0, lon0, r_km):
    """Events whose epicenter lies within r_km of the point (lat0, lon0), earliest first.

    df      : pandas DataFrame with at least the columns origin_utc, latitude, longitude (degrees)
    lat0    : float, latitude of the center in decimal degrees
    lon0    : float, longitude of the center in decimal degrees
    r_km    : float, radius in kilometers (an event exactly at r_km counts as inside)
    returns : pandas DataFrame, the rows of df within the radius with all of df's columns, sorted by
              origin_utc from earliest to latest (an empty DataFrame with the same columns when no event is
              close enough); df itself is not changed
    """
    # Step 1: the distance from the center to every event, with your distance function, in one call.
    # Step 2: keep the rows that are close enough, then order them by time.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T4: write your code here")

In [ ]:
st.check("M01-T4", func=events_within)

In [ ]:
near = events_within(cat, EXAMPLE_LAT, EXAMPLE_LON, EXAMPLE_RADIUS_KM)
print(f"{len(near)} catalog events within {EXAMPLE_RADIUS_KM} km of ({EXAMPLE_LAT}, {EXAMPLE_LON})")
print(near.groupby(near["origin_utc"].dt.year).size().tail(N_SHOW))
near.nlargest(N_SHOW, "ml")[["event_id", "origin_kst", "latitude", "longitude", "depth_km", "ml"]]

**Your selection as a picture.** Every dot is a row that your function returned. The left panel shows when it
happened and how large it was, the right panel where. A dense patch can be an earthquake sequence, a change in
detection, or both; M04 and M05 show how to tell them apart.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
axes[0].scatter(near["origin_utc"], near["ml"], s=9, alpha=0.45, color="tab:blue")
axes[0].set(xlabel="Origin time (UTC)", ylabel="Local magnitude ML", title="Events returned by your function")
axes[0].tick_params(axis="x", rotation=35)
points = axes[1].scatter(near["longitude"], near["latitude"], c=near["ml"], s=12, alpha=0.65, cmap="viridis")
axes[1].plot(EXAMPLE_LON, EXAMPLE_LAT, "r*", ms=12, label="Selection center")
axes[1].set(xlabel="Longitude (°E)", ylabel="Latitude (°N)", title=f"Epicenters within {EXAMPLE_RADIUS_KM:g} km")
axes[1].set_aspect(1 / np.cos(np.radians(EXAMPLE_LAT)))
axes[1].legend(fontsize=8)
fig.colorbar(points, ax=axes[1], label="Local magnitude ML")
show_figure(fig, "M01_selected_catalog.png")

**Explain:** The check leaves out the events within 0.01 km of the circle. Why?

## Part 5 · A one-hour mystery in old records (about 60 min)

A colleague made KST labels for every event, to match the events with a Korean report that lists KST times. The
colleague's cell runs without an error, and the labels look fine. Can they be trusted?

### M01-T5 · A one-hour mystery in old records (debug the trap) · core · about 60 min

The colleague's cell below converts the UTC times to KST with the `'Asia/Seoul'` time zone (`ZoneInfo` from
Python's `zoneinfo` module). Run it, but do not edit it. Then write `kst_mismatch_summary(df)`. It converts
`origin_utc` to KST in two ways, with the fixed offset of task 2 and with `'Asia/Seoul'`, and compares the clock
readings. The docstring gives the two keys it returns.

Example: a table in which every row agrees gives `{"n_wrong": 0, "months": []}`.

In [ ]:
# The colleague's cell. Run it; do not edit it.
from zoneinfo import ZoneInfo

catalog = st.data.kma_catalog()
seoul = ZoneInfo("Asia/Seoul")
catalog["kst_seoul"] = catalog["origin_utc"].dt.tz_convert(seoul)
catalog["kst_label"] = catalog["kst_seoul"].dt.strftime("%Y-%m-%d %H:%M:%S")
catalog[["event_id", "origin_utc", "kst_label"]].tail(3)

In [ ]:
# Before you code: if some of the colleague's labels are wrong, how many do you expect: a handful,
# hundreds or thousands?
st.predict("M01-T5", """

""")

In [ ]:
def kst_mismatch_summary(df):
    """Events whose KST clock reading differs between a fixed +9 h offset and the 'Asia/Seoul' time zone.

    df      : pandas DataFrame with at least the column origin_utc (time-zone-aware, UTC); use no other column
    returns : dict with exactly two keys
        n_wrong : int, the number of rows whose two KST clock readings (date and time of day, to the second)
                  differ
        months  : sorted list of the distinct 'YYYY-MM' of those rows, taken from the fixed-offset reading
                  (an empty list when no row differs)
    df itself is not changed (no new column).
    """
    # Step 1: convert origin_utc to KST in the two ways.
    # Step 2: find the rows whose two clock readings differ, and count them.
    # Step 3: collect the months of those rows (fixed-offset KST), each once, sorted.
    # YOUR CODE HERE
    raise NotImplementedError("M01-T5: write your code here")

In [ ]:
st.check("M01-T5", func=kst_mismatch_summary)

In [ ]:
summary = kst_mismatch_summary(catalog)
print(summary)
in_those_months = catalog["origin_kst"].dt.strftime("%Y-%m").isin(summary["months"])
catalog.loc[in_those_months, ["event_id", "origin_kst", "kst_seoul", "kst_label"]]

**Explain:** Look at the table above. Why are the colleague's labels of those events one hour off?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M01")

**Optional (not checked, about 60 min):**

1. Count the events with ML ≥ `MIN_ML` per year with `groupby`. Why do the early years list fewer small events?
   (Think of the stations.)
2. Set the parameters to the 2017 Pohang mainshock (36.11, 129.37) and 10 km. Run the notebook again and list the
   five largest events near Pohang.
3. Time your `haversine_km` on a million random pairs against a loop with the `math` module.

**Read more:** Harris et al. (2020), NumPy; McKinney (2010), pandas; Wilson et al. (2017), good enough practices
for research code. The [reading list](../../../docs/reading_list.md) has the rest.

Next: M02 (figures and maps).